# Week 2 — CNN + LSTM Image Captioning Model

**Goal:** Build, train, and evaluate a CNN encoder + LSTM decoder captioning model.

### Architecture
```
Image → ResNet50 (frozen) → Linear(256) → h₀ of LSTM
                                              ↓
Caption tokens → Embedding(256) ──────→ LSTM(512) → Linear → Vocab logits
```

In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torchvision.transforms as T

import config
from src.vocabulary import Vocabulary
from src.dataset import load_captions_df, build_dataloaders
from src.lstm_model import CaptioningLSTM
from src.train import train
from src.evaluate import compute_bleu

print('Device:', config.DEVICE)

## 1. Load Data & Vocabulary

In [ ]:
df    = load_captions_df(config.CAPTIONS_FILE)
vocab = Vocabulary.load(os.path.join(config.MODELS_DIR, 'vocab.pkl'))

train_loader, val_loader, test_loader = build_dataloaders(df, vocab, config.IMAGES_DIR)
print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}')

## 2. Build LSTM Model

In [ ]:
model = CaptioningLSTM(vocab_size=len(vocab))
print(model)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {total:,}')

## 3. Test Forward Pass

In [ ]:
images, captions, lengths = next(iter(train_loader))
print(f'Images  shape: {images.shape}')
print(f'Captions shape: {captions.shape}')

model.eval()
with torch.no_grad():
    logits = model(images, captions)
print(f'Output logits shape: {logits.shape}  (B x T-1 x VocabSize)')

## 4. Train Model

In [ ]:
history = train(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    save_name    = 'lstm',
    epochs       = config.NUM_EPOCHS,
    lr           = config.LEARNING_RATE,
    device       = config.DEVICE,
)

## 5. Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
epochs  = range(1, len(history['train_loss'])+1)
ax.plot(epochs, history['train_loss'], 'o-', label='Train Loss', color='#7c6aff', lw=2)
ax.plot(epochs, history['val_loss'],   's--', label='Val Loss',   color='#ff6b9d', lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('LSTM Training Loss Curves')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'lstm_loss.png'), dpi=120)
plt.show()

## 6. BLEU Score Evaluation

In [ ]:
bleu = compute_bleu(model, test_loader, vocab, device=config.DEVICE)

## 7. Qualitative Results

In [ ]:
from src.dataset import load_captions_df
import torchvision.transforms as T

model.eval()
preprocess = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

df_test = load_captions_df(config.CAPTIONS_FILE)
samples = df_test.drop_duplicates('image').sample(6, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (_, row) in zip(axes.flat, samples.iterrows()):
    img = Image.open(os.path.join(config.IMAGES_DIR, row['image'])).convert('RGB')
    tensor = preprocess(img).unsqueeze(0).to(config.DEVICE)
    with torch.no_grad():
        gen = model.caption(tensor, vocab)[0]
    ax.imshow(img)
    ax.set_title(f'Generated:\n{gen}\n\nReference:\n{row["caption"][:60]}',
                 fontsize=7)
    ax.axis('off')

plt.suptitle('LSTM Captioning Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'lstm_results.png'), dpi=120)
plt.show()

print('Week 2 complete ✓')